1. Stage 1 (Translation & Query Expansion) $\rightarrow$ Claude-Sonnet-4.6 (Best at understanding the medical scope and generating dense queries).
2. Stage 2 (Information Extraction & Synthesis) $\rightarrow$ Gemini-2.5-Flash (The absolute king of Faithfulness at 0.729; it strictly reads the context without making things up).
3. Stage 3 (Answer Refinement & Formatting) $\rightarrow$ GPT-4o-mini (The highest Answer Relevancy score; it takes Gemini's rigid factual block and turns it into a perfectly polished answer).

##Install Dependencies

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB

##The VertexAI Fix

In [ ]:
import sys
import types

class DummyVertexAI: pass
class DummyChatVertexAI: pass

dummy_llms = types.ModuleType("langchain_community.llms")
dummy_llms.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms"] = dummy_llms

dummy_chat_models = types.ModuleType("langchain_community.chat_models")
dummy_chat_models.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models"] = dummy_chat_models

dummy_chat_vertexai = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertexai.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertexai

dummy_llms_vertexai = types.ModuleType("langchain_community.llms.vertexai")
dummy_llms_vertexai.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms.vertexai"] = dummy_llms_vertexai

##System Setup, Database Connection & Prompts

In [ ]:
import os
import json
import pandas as pd
import time
import nest_asyncio
from google.colab import userdata, drive
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate

nest_asyncio.apply()
drive.mount('/content/drive')

# 1. SETUP PATHS & CREDENTIALS
DRIVE_PATH = '/content/drive/MyDrive/'
DATASET_PATH = DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv'
df = pd.read_csv(DATASET_PATH)

NEON_CONNECTION_STRING = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# 2. INITIALIZE VECTOR STORE
print("Connecting to original PGVector Database...")
bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device': 'cpu'})

vector_store = PGVector(
    embeddings=bge_embeddings,
    collection_name="awmf_baseline_bge",
    connection=NEON_CONNECTION_STRING,
    use_jsonb=True
)
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# 3. INITIALIZE OUR SPECIALIZED MODELS
print("Initializing the Specialists...")
claude_stage1 = ChatOpenAI(model="anthropic/claude-sonnet-4.6", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0)
gemini_stage2 = ChatOpenAI(model="google/gemini-2.5-flash", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0)
gpt_stage3    = ChatOpenAI(model="openai/gpt-4o-mini", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0)

# 4. THREE-STAGE PIPELINE PROMPTS
stage1_expansion_prompt = PromptTemplate(
    template="""You are an expert medical search term generator.
First, translate the following English medical question into German.
Then, generate 3 to 4 highly formal German clinical synonyms, related medical conditions, or official MeSH terms (Query Expansion) that would likely appear in a formal clinical guideline.
Output ONLY the translated question and the synonyms combined as a single continuous search string. Do not include bullet points, labels, or introductory text.

English Question:
{question}""",
    input_variables=["question"]
)

stage2_extraction_prompt = PromptTemplate(
    template="""You are an expert medical data extractor. Read the provided German clinical guidelines and extract ALL clinical facts, criteria, or medical recommendations necessary to answer the English medical question.
Your response must be written in English.
Be highly technical, literal, and strict. Do NOT assume, extrapolate, or add external clinical knowledge. If the text does not contain the answer, state 'Information not found'.

Context (German):
{context}

Question (English):
{question}

Extracted English Facts:""",
    input_variables=["context", "question"]
)

stage3_polish_prompt = PromptTemplate(
    template="""You are a medical AI communicator translating raw extraction data into a clean clinical answer.
Take the following raw extracted medical facts and format them into a highly accurate, professional, and readable medical summary answering the user's question.
Maintain complete fidelity to the facts provided. Do not add external recommendations.

Raw Extracted Facts:
{raw_facts}

Original Question:
{question}

Final Answer (English):""",
    input_variables=["raw_facts", "question"]
)

print("Compound system initialized successfully!")

Mounted at /content/drive
Connecting to original PGVector Database...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Initializing the Specialists...
Compound system initialized successfully!


## The Compound Execution Loop

In [ ]:
output_file = f"{DRIVE_PATH}COMPOUND_SYSTEM_results.json"
results = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("RUNNING COMPOUND PIPELINE EXPERIMENT (200 QUESTIONS)...")

for index, row in df.iterrows():
    english_question = row['English_Open_Question']
    english_ground_truth = row['English_Correct_Text']

    try:
        # ---- STAGE 1: CLAUDE TRANSLATES & EXPANDS ----
        f_stage1 = stage1_expansion_prompt.format(question=english_question)
        expanded_german_query = claude_stage1.invoke(f_stage1).content.strip()

        # ---- DATABASE RETRIEVAL ----
        retrieved_docs = retriever.invoke(expanded_german_query)
        contexts = [doc.page_content for doc in retrieved_docs]
        context_string = "\n\n".join(contexts)

        # ---- STAGE 2: GEMINI EXTRACTS STRICT FACTS ----
        f_stage2 = stage2_extraction_prompt.format(context=context_string, question=english_question)
        raw_facts = gemini_stage2.invoke(f_stage2).content.strip()

        # ---- STAGE 3: GPT-4o-MINI REFINES AND POLISHES ----
        f_stage3 = stage3_polish_prompt.format(raw_facts=raw_facts, question=english_question)
        final_polished_answer = gpt_stage3.invoke(f_stage3).content.strip()

        # ---- SAVE PIPELINE OUTPUTS ----
        results["question"].append(english_question)
        results["answer"].append(final_polished_answer)
        results["contexts"].append(contexts)
        results["ground_truth"].append(english_ground_truth)

        with open(output_file, 'w') as f:
            json.dump(results, f)

        if (index + 1) % 20 == 0:
            print(f"Progress: {index + 1}/{len(df)}")
            print(f"Sample Live Pipeline Pass for Q{index+1}:")
            print(f" -> Claude Search Query: {expanded_german_query[:80]}...")
            print(f" -> Gemini Facts Len: {len(raw_facts)} chars")
            print(f" -> GPT Polished Answer: {final_polished_answer[:100]}...\n")

        time.sleep(2) # Protect API limits across multiple providers

    except Exception as e:
        print(f"Error at index {index}: {e}")
        time.sleep(5)
        continue

print("Compound Pipeline Run Complete! Output saved.")

RUNNING COMPOUND PIPELINE EXPERIMENT (200 QUESTIONS)...
Progress: 20/200
Sample Live Pipeline Pass for Q20:
 -> Claude Search Query: Ein 48-jähriger Mann wird wegen plötzlich einsetzender Atemnot seit 6 Stunden in...
 -> Gemini Facts Len: 3333 chars
 -> GPT Polished Answer: In this case, the 48-year-old man presents with symptoms and clinical signs suggestive of decompensa...

Progress: 40/200
Sample Live Pipeline Pass for Q40:
 -> Claude Search Query: Ein 59-jähriger Mann wird wegen progressiver Gelenkschmerzen untersucht. Es best...
 -> Gemini Facts Len: 3353 chars
 -> GPT Polished Answer: The clinical presentation of the 59-year-old man, characterized by progressive joint pain, swelling,...

Progress: 60/200
Sample Live Pipeline Pass for Q60:
 -> Claude Search Query: Ein 7-jähriger Junge wird zur Nachsorgeuntersuchung beim Kinderarzt vorgestellt....
 -> Gemini Facts Len: 222 chars
 -> GPT Polished Answer: The experimental therapy being considered for the 7-year-old boy targets media

##Ragas Evaluation

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

print("GRADING THE COMPOUND PIPELINE PERFORMANCE...")

with open(output_file, 'r') as f:
    data = json.load(f)

evaluation_features = Features({
    "question": Value("string"),
    "answer": Value("string"),
    "contexts": Sequence(Value("string")),
    "ground_truth": Value("string"),
})

eval_dataset = Dataset.from_dict(data, features=evaluation_features)

# Keeping the referee exactly consistent (GPT-4o-mini baseline judge)
judge_llm = LangchainLLMWrapper(gpt_stage3)
ragas_embeddings = LangchainEmbeddingsWrapper(bge_embeddings)

eval_results = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
    llm=judge_llm,
    embeddings=ragas_embeddings,
    run_config=RunConfig(timeout=300, max_workers=2, max_retries=5)
)

res_df = eval_results.to_pandas()
res_df.to_csv(f"{DRIVE_PATH}COMPOUND_SYSTEM_FINAL_EVALUATION.csv", index=False)

print("\nEXPERIMENT COMPLETE!")
print("Here are the performance scores for the Compound AI System:")
print(res_df[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().round(3))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_4178/3930215768.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_4178/3930215768.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

GRADING THE COMPOUND PIPELINE PERFORMANCE...


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]


EXPERIMENT COMPLETE!
Here are the performance scores for the Compound AI System:
context_precision    0.175
context_recall       0.225
faithfulness         0.262
answer_relevancy     0.546
dtype: float64
